# Day 73 — Activation Functions

This notebook covers The Linearity Trap, Mathematical Proof of Linear Collapse, Core Activation Functions (Sigmoid, Tanh, ReLU, Softmax), Zero-Centering Optimization Dynamics, and NumPy Implementation

## Validation goals

- Understand the linearity trap and why multi-layer neural networks collapse into single linear models without non-linear activation functions
- Master the mathematical definitions, output ranges, and primary use cases of Sigmoid, Tanh, ReLU, and Softmax activation functions
- Implement core activation functions from scratch in Python using vectorized NumPy operations
- Analyze the optimization advantages of zero-centered outputs ($\text{Tanh}$) and gradient preservation ($\text{ReLU}$)

### 1. The Linearity Trap: Why Activation Functions Are Mandatory

Suppose you build a deep neural network with 10 hidden layers, but every single neuron simply passes its raw weighted sum straight through without any modification ($f(z) = z$).

Mathematically, multiplying matrices of linear equations together (Layer 1 $\times$ Layer 2 $\times$ Layer 3) collapses entirely into a single linear equation. No matter how many hidden layers you add, a purely linear network cannot solve non-linear problems (like separating classes distributed in a circle or capturing complex retail demand curves).

Activation functions introduce non-linearity, allowing neural networks to approximate arbitrarily complex functions.

### 2. The Core Activation Functions

Here are the four industry-standard activation functions you will encounter in deep learning engineering:

**A. Sigmoid Function**

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
- Range: $(0, 1)$
- Primary Use Case: Historically used in hidden layers and currently standard for the output layer of binary classification models (representing probabilities).
- Limitation: Suffers from the vanishing gradient problem for very high or very low inputs, where the gradient approaches zero, halting learning during backpropagation.

**B. Tanh (Hyperbolic Tangent) Function**

$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$
- Range: $(-1, 1)$
- Primary Use Case: Hidden layers. Because it is zero-centered (outputs are centered around zero), it often helps convergence faster than the sigmoid function.
- Limitation: Still suffers from vanishing gradients at extreme positive or negative inputs.

**C. ReLU (Rectified Linear Unit)**

$$\text{ReLU}(z) = \max(0, z)$$
- Range: $[0, \infty)$
- Primary Use Case: The default standard for hidden layers in modern deep learning architectures. It is computationally efficient (just a threshold check) and does not saturate for positive values, mitigating the vanishing gradient problem.
- Limitation: Dying ReLU problem—if a large gradient flows through a neuron such that its weights update and it outputs negative values for all inputs, the gradient becomes zero, and the neuron permanently stops firing.

**D. Softmax Function**

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}}$$
- Range: $(0, 1)$, with all output values summing up to $1.0$.
- Primary Use Case: Used exclusively in the output layer for multi-class classification tasks to convert raw scores (logits) into a probability distribution.

### 3. Implementing Activation Functions from Scratch in Python (NumPy)

Let's write a clean Python script implementing the mathematical definitions of Sigmoid, Tanh, and ReLU using NumPy.

In [1]:
import numpy as np

# 1. Generate an array of sample pre-activation values (logits)
z_values = np.array([-3.5, -1.0, 0.0, 1.5, 4.2])

# 2. Sigmoid Activation
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# 3. Tanh Activation
def tanh(z):
    return np.tanh(z)

# 4. ReLU Activation
def relu(z):
    return np.maximum(0, z)

# Execute activations
print("--- Day 73: Activation Functions Demonstration ---")
print(f"Raw Inputs (Z): {z_values}\n")
print(f"Sigmoid Output: {np.round(sigmoid(z_values), 4)}")
print(f"Tanh Output:    {np.round(tanh(z_values), 4)}")
print(f"ReLU Output:    {np.round(relu(z_values), 4)}")

--- Day 73: Activation Functions Demonstration ---
Raw Inputs (Z): [-3.5 -1.   0.   1.5  4.2]

Sigmoid Output: [0.0293 0.2689 0.5    0.8176 0.9852]
Tanh Output:    [-0.9982 -0.7616  0.      0.9051  0.9996]
ReLU Output:    [0.  0.  0.  1.5 4.2]


### Guided Question:

Explain what would happen to the expressive power of a 5-layer neural network if every hidden layer used a purely linear activation function ($f(z) = z$).

### Answer:

If every hidden layer of a 5-layer neural network uses a purely linear activation function ($f(z) = z$), the expressive power of the network collapses entirely.

Mathematically, the deep network becomes completely equivalent to a single-layer linear model (such as standard linear regression or a single affine transformation), rendering the additional layers functionally useless despite the increased parameter count and computational cost.

**The Mathematical Proof of Collapse**

To see why this happens, let's trace the forward pass of a multi-layer network with linear activations.

Let $\mathbf{x}$ be the input vector, and let each layer perform a linear transformation defined by weight matrix $\mathbf{W}^{(l)}$ and bias vector $\mathbf{b}^{(l)}$.

- Layer 1:$$\mathbf{h}^{(1)} = \mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$$
- Layer 2:$$\mathbf{h}^{(2)} = \mathbf{W}^{(2)}\mathbf{h}^{(1)} + \mathbf{b}^{(2)} = \mathbf{W}^{(2)}(\mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)}) + \mathbf{b}^{(2)}$$$$\mathbf{h}^{(2)} = (\mathbf{W}^{(2)}\mathbf{W}^{(1)})\mathbf{x} + (\mathbf{W}^{(2)}\mathbf{b}^{(1)} + \mathbf{b}^{(2)})$$

If you continue this process through all 5 layers, matrix multiplication is associative. The entire sequence of matrix multiplications and bias additions collapses into a single effective weight matrix $\mathbf{W}_{\text{eff}}$ and a single effective bias vector $\mathbf{b}_{\text{eff}}$:$$\mathbf{y} = \mathbf{W}_{\text{eff}}\mathbf{x} + \mathbf{b}_{\text{eff}}$$Where:$$\mathbf{W}_{\text{eff}} = \mathbf{W}^{(5)} \mathbf{W}^{(4)} \mathbf{W}^{(3)} \mathbf{W}^{(2)} \mathbf{W}^{(1)}$$

No matter how many hidden layers you add ($5$, $50$, or $1,000$), the composition of linear functions is always just another linear function.

**Practical Consequences on Expressive Power**

- Inability to Learn Non-Linear Boundaries: Linear models can only draw flat hyperplanes to separate data. They cannot solve non-linear problems (such as the classic XOR problem) or approximate complex curves, curved decision boundaries, or multi-modal distributions.
- Loss of the Benefits of Depth: In standard deep learning, depth allows networks to learn hierarchical representations (e.g., edges $\rightarrow$ shapes $\rightarrow$ objects in computer vision). Without non-linearities, stacking layers provides no hierarchical feature extraction—every layer is just scaling and rotating the input space linearly.
- Wasted Computational Resources: While a 5-layer linear network requires significantly more memory to store parameters and compute matrix multiplications during forward and backward passes, its hypothesis space (the set of functions it can represent) is identical to a shallow network with zero hidden layers.
- Optimization and Gradient Pathologies: Although linear deep networks do not lose representation capacity, training them via gradient descent introduces severe challenges. Deep linear networks suffer from ill-conditioned optimization landscapes, creating vanishing or exploding gradients through repeated matrix multiplications, making them notoriously difficult to train compared to shallow linear models.

**Summary**

Non-linear activation functions (such as ReLU, GELU, Tanh, or Sigmoid) are not optional accessories in deep learning; they are the fundamental engine of a neural network's power. Without them, the Universal Approximation Theorem does not hold, and a deep network loses all advantages over basic linear regression.

### Guided Question:

What is the primary advantage of the Tanh function over the Sigmoid function regarding output centering, and why is ReLU generally preferred over both in deep hidden layers?

### Answer:

**Tanh vs. Sigmoid: The Advantage of Zero-Centering**

The primary advantage of the $\text{Tanh}$ function over the Sigmoid function lies in its output range and zero-centered symmetry:
- Sigmoid Output Range: $(0, 1)$ — Strictly positive.
- Tanh Output Range: $(-1, 1)$ — Zero-centered (symmetric around $0$).

**Why Zero-Centering Matters for Optimization**

When a neuron uses a strictly positive activation function like Sigmoid, all of its outputs are positive. During backpropagation, the gradients of the loss with respect to the incoming weights ($\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a} \cdot f'(z) \cdot x$) depend directly on the sign of the input $x$.

If all incoming activations $x$ are strictly positive, the gradients for all weights feeding into that neuron will share the exact same sign. This forces the gradient descent algorithm to update weights in a biased, "zigzagging" path (e.g., all weights must increase together or decrease together), which significantly slows down convergence and makes optimization inefficient.Because $\text{Tanh}$ outputs both positive and negative values, the mean activation across a layer is closer to zero. 

This allows incoming gradients to have mixed signs, enabling weight updates to move in more direct, balanced trajectories toward the minimum of the loss surface.

**Why ReLU is Generally Preferred in Deep Hidden Layers**

Despite $\text{Tanh}$ solving the zero-centering problem, both Sigmoid and $\text{Tanh}$ are rarely used in deep hidden layers today. The ReLU (Rectified Linear Unit) function, defined as:$$f(z) = \max(0, z)$$has become the standard default for deep hidden layers due to several key advantages:

**1. Mitigation of the Vanishing Gradient Problem**

- Sigmoid and Tanh: Both functions saturate at their extremes (Sigmoid saturates at $0$ and $1$; $\text{Tanh}$ saturates at $-1$ and $1$). In these saturation zones, the derivative (slope) approaches zero ($f'(z) \approx 0$). In a deep network with many layers, multiplying these small derivatives together during backpropagation causes gradients to shrink exponentially toward zero, halting learning entirely (the vanishing gradient problem).
- ReLU: For all positive inputs ($z > 0$), the derivative of ReLU is a constant $1.0$. Because the gradient does not shrink as activations become large, it flows unimpeded through positive layers, allowing much deeper networks to be trained effectively.

**2. Computational Efficiency**

Calculating Sigmoid and $\text{Tanh}$ requires evaluating exponential functions ($\frac{1}{1 + e^{-z}}$ and hyperbolic equivalents), which are computationally expensive floating-point operations. ReLU, by contrast, is a simple thresholding operation ($\max(0, z)$) that can be computed instantly in hardware with minimal CPU/GPU overhead.
**3. Biological Plausibility and Sparsity**

ReLU introduces sparsity into the network. For any negative input, ReLU outputs an exact zero ($f(z) = 0$), meaning that neuron does not activate or contribute to the downstream computation. This results in sparse representations where only a subset of neurons fire for a given input, making the network more efficient and better at disentangling features.

### Guided Question:

In the NumPy snippet above, why does `np.maximum(0, z)` effectively implement the `ReLU` activation function?

### Answer:

The function np.maximum(0, z) implements the Rectified Linear Unit (ReLU) because ReLU is formally defined as:$$f(z) = \max(0, z)$$

In NumPy, `np.maximum()` performs an element-wise comparison between the scalar 0 and each element in the array z, effectively handling both negative and positive values according to the piecewise rules of ReLU:

- For negative inputs ($z < 0$): The maximum between 0 and a negative number is 0 (e.g., np.maximum(0, -3.5) evaluates to 0.0). This "turns off" negative activations, introducing sparsity.
- For non-negative inputs ($z \ge 0$): The maximum between 0 and a positive number or zero is the number itself (e.g., np.maximum(0, 4.2) evaluates to 4.2). This passes positive pre-activation values through completely unchanged, preserving the gradient during backpropagation.Because NumPy vectorizes this operation across the entire array at the C level, it computes the piecewise thresholds for every element simultaneously without requiring explicit Python for loops.